# frame 360

In [ ]:
import pandas as pd
import json
import ast  # for safely parsing stringified lists

# --- Load the CSV ---
df = pd.read_csv("./frames_360_by_match/frames_360_3788741.csv")

# --- Prepare result list ---
result = []

# --- Group by event id ---
for event_id, group in df.groupby("id"):
    # Parse visible area (same for all rows)
    visible_area_str = group["visible_area"].iloc[0]
    visible_area = ast.literal_eval(visible_area_str) if pd.notna(visible_area_str) else []

    # Build freeze frame list
    freeze_frame = []
    for _, row in group.iterrows():
        location = ast.literal_eval(row["location"]) if pd.notna(row["location"]) else [None, None]
        freeze_frame.append({
            "teammate": bool(row["teammate"]),
            "actor": bool(row["actor"]),
            "keeper": bool(row["keeper"]),
            "location": location
        })

    # Append one record per event
    result.append({
        "event_uuid": event_id,
        "visible_area": visible_area,
        "freeze_frame": freeze_frame
    })

# --- Write output JSON ---
with open("open-data/data/three-sixty/3788741.json", "w") as f:
    json.dump(result, f, indent=2)


# event

In [ ]:
import pandas as pd
import json
import ast

def auto_convert(v: str):
    """Convert numeric/boolean-like strings to proper types when safe."""
    if v is None:
        return None
    s = str(v).strip()
    if s == "":
        return None
    if s.lower() in {"true", "false"}:
        return s.lower() == "true"
    if s.isdigit() or (s.startswith("-") and s[1:].isdigit()):
        try:
            return int(s)
        except Exception:
            pass
    try:
        # Avoid converting timestamps like '00:00:12.345'
        if ":" not in s:
            return float(s)
    except Exception:
        pass
    return v

def safe_parse(v):
    """Parse list/dict strings; otherwise auto-convert scalars; None for NaN/empty."""
    if v is None:
        return None
    if isinstance(v, float) and pd.isna(v):
        return None
    s = str(v).strip()
    if s == "" or s.lower() == "nan":
        return None
    if (s.startswith("[") and s.endswith("]")) or (s.startswith("{") and s.endswith("}")):
        try:
            return ast.literal_eval(s)
        except Exception:
            pass
    return auto_convert(s)

def make_id_name_obj(id_val, name_val):
    id_v = safe_parse(id_val)
    name_v = safe_parse(name_val)
    if id_v is None and name_v is None:
        return None
    obj = {}
    if id_v is not None:
        obj["id"] = id_v
    if name_v is not None:
        obj["name"] = name_v
    return obj if obj else None

def clean_empty(obj):
    """Recursively drop None/empty containers."""
    if isinstance(obj, dict):
        out = {}
        for k, v in obj.items():
            cv = clean_empty(v)
            if cv is None:
                continue
            if isinstance(cv, (dict, list)) and len(cv) == 0:
                continue
            out[k] = cv
        return out if out else None
    if isinstance(obj, list):
        out = []
        for x in obj:
            cx = clean_empty(x)
            if cx is None:
                continue
            if isinstance(cx, (dict, list)) and len(cx) == 0:
                continue
            out.append(cx)
        return out if out else None
    return obj

def assign_id_name(container: dict, base: str, field_name: str, value):
    """
    Ensure container[base] is a dict; if a scalar lives there already,
    upgrade it to {'name': <old>} before setting 'id'/'name'.
    """
    existing = container.get(base)
    if not isinstance(existing, dict):
        if existing is None:
            container[base] = {}
        else:
            container[base] = {"name": existing}
    container[base][field_name] = value

def convert_events(csv_path: str, out_json_path: str):
    df = pd.read_csv(csv_path, dtype=str)
    events = []
    core_cols = {
        "id", "index", "index_num", "period", "timestamp", "minute", "second",
        "possession", "possession_team_id", "possession_team",
        "team_id", "team", "player_id", "player", "position",
        "location", "duration", "type", "play_pattern", "related_events",
        "tactics"
    }

    for _, row in df.iterrows():
        ev = {}

        # id + index (some csvs use index_num)
        if pd.notna(row.get("id", None)):
            ev["id"] = str(row["id"])
        idx = row.get("index")
        if not idx or str(idx).strip().lower() == "nan":
            idx = row.get("index_num")
        if idx and str(idx).strip().lower() != "nan":
            ev["index"] = safe_parse(idx)

        # simple core fields
        for fld in ["period", "timestamp", "minute", "second", "duration"]:
            if fld in row and pd.notna(row[fld]):
                ev[fld] = safe_parse(row[fld])

        # typed cores
        if pd.notna(row.get("type", None)):
            ev["type"] = {"name": str(row["type"])}
        if pd.notna(row.get("play_pattern", None)):
            ev["play_pattern"] = {"name": str(row["play_pattern"])}

        # team contexts
        poss_team = make_id_name_obj(row.get("possession_team_id"), row.get("possession_team"))
        if poss_team:
            ev["possession_team"] = poss_team
        team_obj = make_id_name_obj(row.get("team_id"), row.get("team"))
        if team_obj:
            ev["team"] = team_obj
        player_obj = make_id_name_obj(row.get("player_id"), row.get("player"))
        if player_obj:
            ev["player"] = player_obj

        if pd.notna(row.get("position", None)):
            ev["position"] = {"name": str(row["position"])}
        if pd.notna(row.get("possession", None)):
            ev["possession"] = safe_parse(row["possession"])
        if pd.notna(row.get("location", None)):
            ev["location"] = safe_parse(row["location"])
        if pd.notna(row.get("related_events", None)):
            ev["related_events"] = safe_parse(row["related_events"])
        if pd.notna(row.get("tactics", None)):
            ev["tactics"] = safe_parse(row["tactics"])

        # dynamic: every other non-null column → nested by prefix
        for col in row.index:
            if col in core_cols:
                continue
            val = row[col]
            if not val or str(val).strip().lower() == "nan":
                continue
            parsed = safe_parse(val)

            parts = col.split("_", 1)
            if len(parts) == 2:
                prefix, subkey = parts[0], parts[1]
                # make sure we have a dict at ev[prefix]
                prev = ev.get(prefix)
                if not isinstance(prev, dict):
                    ev[prefix] = {}
                    if prev is not None and not isinstance(prev, dict):
                        ev[prefix]["name"] = prev  # upgrade scalar to dict

                # subkey with id/name pairing
                if subkey.endswith("_id") or subkey.endswith("_name"):
                    base = subkey.rsplit("_", 1)[0]
                    field_name = subkey.split("_")[-1]
                    assign_id_name(ev[prefix], base, field_name, parsed)
                else:
                    existing = ev[prefix].get(subkey)
                    # shallow-merge dicts; otherwise overwrite
                    if isinstance(existing, dict) and isinstance(parsed, dict):
                        ev[prefix][subkey] = {**existing, **parsed}
                    else:
                        ev[prefix][subkey] = parsed
            else:
                # no underscore → put at top-level
                ev[col] = parsed

        ev = clean_empty(ev)
        if ev:
            events.append(ev)

    with open(out_json_path, "w") as f:
        json.dump(events, f, indent=2)

    return out_json_path

# Usage example:
convert_events("./events_by_match/events_3788741.csv", "open-data/data/events/3788741.json")
